# FHR-TD3 on Pendulum

`Pendulum-v1`, the validation family of the track: 20k steps, two seeds, every code path (learned c, c(s,a), PER, frozen-theory c) and both probes on a dense cadence. Stock SB3 TD3 vs **FHRTD3** on the recipe in
`configs/config_sb3_td3.yaml` (20,000 steps, seeds [44, 66]).

**Penalty scale (TD3-native).** Stock TD3's critic loss is the plain sum over
the twin critics, $\sum_i \mathrm{MSE}_i$, and the FHR term joins on the same
footing: $L = \sum_i \mathrm{MSE}_i + \lambda \sum_i \mathrm{Huber}_i$ - per
critic $\mathrm{MSE}_i + \lambda\,\mathrm{Huber}_i$, with no division by the
number of critics. `penalty_raw` logs $\sum_i \mathrm{Huber}_i$ and `td_loss`
the stock $\sum_i \mathrm{MSE}_i$, so $\rho_{loss} = \lambda\,$penalty / TD is
the ratio of the two terms exactly as optimised. $\lambda$ is therefore a
TD3-scale knob and **not numerically comparable to the SAC families'**
$\lambda$; the cross-algorithm quantities are the stream ratios of section 5b.

**Why TD3.** The target is the plain Bellman backup (no entropy term with a
moving temperature inside the value the recurrence is fitted to), so the
on-trajectory recurrence is exact up to the zero-mean smoothing noise and the
frozen-theory control $c = (1 + 1/\gamma, -1/\gamma)$ is a genuine identity
test. The actor is deterministic and trained on $Q_1(s, \pi(s))$ alone every
`policy_delay` critic steps, so the FHR-shaped $Q_1$ feeds straight into the
deterministic policy gradient. Exploration is a fixed action noise, so
`rewards.csv` is the noisy behaviour policy while `eval.csv` is the
deterministic actor.


The **baseline arm is bit-for-bit stock SB3 TD3** - same RNG stream, same
updates, with both probes on (asserted in `tests/test_sb3_td3_fhr.py`) - so
an arm-vs-baseline gap is attributable to the FHR penalty alone. Every arm
below is exactly what `configs/config_sb3_td3.yaml` defines under `experiment.fhr_experiments`.

**Every figure is saved individually** under `figures/sb3_pendulum_td3/` (PDF + PNG,
300 dpi) so each can be copy-pasted on its own; the file name is printed under
each cell.

## 0 · Launch - train whatever the config defines

In [1]:
import pathlib, sys
SRC_RUNNERS = pathlib.Path.cwd().parent / "src"
if str(SRC_RUNNERS) not in sys.path:
    sys.path.insert(0, str(SRC_RUNNERS))
import run_sb3_seeds as runner
import yaml

CONFIG = "configs/config_sb3_td3.yaml"
MANIFEST = "cached/sb3_runs_manifest_td3.json"
# Every arm below - launched and analysed - is exactly what the config's
# experiment.fhr_experiments block currently defines; nothing is hardcoded.
EXPERIMENTS = sorted(int(k) for k in
                     (yaml.safe_load(open(CONFIG))["experiment"]
                      .get("fhr_experiments") or {}))
print("config experiments:", EXPERIMENTS)

LAUNCH = False        # the waves are launched detached from a shell (README);
FORCE_EXP = False     # flip to launch (or resume the missing pairs) from here
if LAUNCH:
    manifest = runner.launch_all(max_workers=6, force=FORCE_EXP,
                                 experiments=EXPERIMENTS, config=CONFIG)
    print({k: sorted(v) for k, v in manifest["runs"].items()})

config experiments: [1, 2, 3, 4, 5]


## Setup - the figure toolkit

Every figure in this notebook comes from
[`analysis.visualisations.fhr_figures`](../../../src/analysis/visualisations/fhr_figures.py),
so every family plots the same way and a fix lands once. Conventions:

* **Colour identifies the arm**, never a single hyper-parameter; the baseline
  is near-black; **dashed = frozen-c control**, dotted = c(s,a), dash-dot = PER.
* **The seed band is mean ± 1 s.e.m.** (`ff.BAND`: "sem" | "ci95" | "iqr" | "minmax").
* **Sample efficiency is read off the training stream** (`rewards.csv`), not
  the greedy-eval curve.
* **One figure per cell, saved individually** (`F.save` prints the paths).

In [2]:
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "analysis").is_dir())
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from analysis.visualisations import fhr_figures as ff

ff.set_pub_style()
ff.BAND = "sem"          # seed band: "sem" | "ci95" | "iqr" | "minmax"

F = ff.load_family(CONFIG, MANIFEST)
F.summary()

# The sample-efficiency ladder (training stream, section 3). Pass step= /
# start= to auto_thresholds to pin it explicitly.
THRESHOLDS = F.auto_thresholds()
print("thresholds:", THRESHOLDS)

sb3_pendulum_td3  (Pendulum-v1)   seeds 44, 66
  baseline  baseline                   2 runs   #222222
  exp1      $\lambda$0.1$\cdot$r2      2 runs   #0072B2
  exp2      $\lambda$1$\cdot$r2        2 runs   #D55E00
  exp5      $\lambda$1$\cdot$r2 frozen-c 2 runs   #009E73
  exp3      $\lambda$0.1$\cdot$r2 c(s,a) shared 2 runs   #CC79A7
  exp4      $\lambda$0.1$\cdot$r2 +PER 2 runs   #56B4E9
thresholds: [-500.0, -400.0, -300.0, -200.0]


## 1 · Training curves - the episodes as trained

`rewards.csv`: every training episode's return exactly as the behaviour
policy experienced it (exploration noise included), against cumulative env
steps, rolling-mean smoothed. One figure per arm against the shared baseline,
then the overlay. Dots mark where the seed-mean curve first crosses each
threshold of the sample-efficiency ladder (tabulated in section 3).

In [3]:
for a in F.fhr_arms:
    fig = F.fig_training(a.key, thresholds=THRESHOLDS)
    F.save(fig, f"01_training_{a.key}")
    plt.show()

saved: figures/sb3_pendulum_td3/01_training_exp1.pdf  figures/sb3_pendulum_td3/01_training_exp1.png


saved: figures/sb3_pendulum_td3/01_training_exp2.pdf  figures/sb3_pendulum_td3/01_training_exp2.png


saved: figures/sb3_pendulum_td3/01_training_exp5.pdf  figures/sb3_pendulum_td3/01_training_exp5.png


saved: figures/sb3_pendulum_td3/01_training_exp3.pdf  figures/sb3_pendulum_td3/01_training_exp3.png


saved: figures/sb3_pendulum_td3/01_training_exp4.pdf  figures/sb3_pendulum_td3/01_training_exp4.png


In [4]:
fig = F.fig_overlay("train")
F.save(fig, "01_training_all")
plt.show()

saved: figures/sb3_pendulum_td3/01_training_all.pdf  figures/sb3_pendulum_td3/01_training_all.png


## 2 · Learning curves - greedy evaluation

`eval.csv`: the deterministic actor on fixed reset seeds, so the curves are
paired across arms and the training stream is untouched. One figure per arm
against the baseline, then the overlay.

In [5]:
for a in F.fhr_arms:
    fig = F.fig_eval(a.key)
    F.save(fig, f"02_eval_{a.key}")
    plt.show()

saved: figures/sb3_pendulum_td3/02_eval_exp1.pdf  figures/sb3_pendulum_td3/02_eval_exp1.png


saved: figures/sb3_pendulum_td3/02_eval_exp2.pdf  figures/sb3_pendulum_td3/02_eval_exp2.png


saved: figures/sb3_pendulum_td3/02_eval_exp5.pdf  figures/sb3_pendulum_td3/02_eval_exp5.png


saved: figures/sb3_pendulum_td3/02_eval_exp3.pdf  figures/sb3_pendulum_td3/02_eval_exp3.png


saved: figures/sb3_pendulum_td3/02_eval_exp4.pdf  figures/sb3_pendulum_td3/02_eval_exp4.png


In [6]:
fig = F.fig_overlay("eval")
F.save(fig, "02_eval_all")
plt.show()

saved: figures/sb3_pendulum_td3/02_eval_all.pdf  figures/sb3_pendulum_td3/02_eval_all.png


## 3 · Sample efficiency - the claim FHR actually makes

For each arm and each threshold: the first env step at which that seed's
rolling-mean training return reaches it, averaged over seeds. A threshold
only some seeds reach is marked (open marker / `*n`) - the mean over the seeds
that made it is biased low and is deliberately kept off the line.

In [7]:
_ = F.table_sample_efficiency(THRESHOLDS)

steps to reach, training stream (rolling 50 eps) - Pendulum-v1, seeds 44, 66
arm                             final eval         -500        -400        -300        -200
--------------------------------------------------------------------------------------------
baseline (stock SB3 TD3)            -136.9         11k          12k          13k          14k  
FHR lam=0.1 r=2                     -136.7         11k          12k          13k          15k  
FHR lam=1 r=2                       -135.6         11k          12k          13k          14k  
FHR lam=1 r=2 frozen-c              -136.3         11k          12k          13k          16k  
FHR lam=0.1 r=2 c(s,a) shared       -134.3         11k          12k          13k          14k  
FHR lam=0.1 r=2 +PER                -134.8         11k          12k          13k          14k  
* = only that many seeds reached the threshold; the mean is over those seeds only.


In [8]:
fig = F.fig_steps_to_threshold(THRESHOLDS)
F.save(fig, "03_steps_to_threshold")
plt.show()

saved: figures/sb3_pendulum_td3/03_steps_to_threshold.pdf  figures/sb3_pendulum_td3/03_steps_to_threshold.png


In [9]:
# The same numbers as a ratio: > 1 means the arm reached that return in fewer
# environment steps than the baseline.
fig = F.fig_speedup(THRESHOLDS)
F.save(fig, "03_speedup")
plt.show()

saved: figures/sb3_pendulum_td3/03_speedup.pdf  figures/sb3_pendulum_td3/03_speedup.png


## 4 · Final performance - one figure per lambda

Final greedy-eval return, one figure per lambda rung. Bar = seed mean,
whisker = ± 1 s.e.m., open dots = the individual seeds, percentage = change
against the baseline of the same figure.

In [10]:
for lam in F.lambdas:
    fig = F.fig_final(lam)
    F.save(fig, f"04_final_lambda{lam:g}")
    plt.show()

saved: figures/sb3_pendulum_td3/04_final_lambda0.1.pdf  figures/sb3_pendulum_td3/04_final_lambda0.1.png


saved: figures/sb3_pendulum_td3/04_final_lambda1.pdf  figures/sb3_pendulum_td3/04_final_lambda1.png


In [11]:
# The learned-c sweep as one lambda x order map (skipped when the family has
# only one lambda or one order).
if len(F.lambdas) > 1 and len(F.orders) > 1:
    fig = F.fig_grid()
    F.save(fig, "04_grid")
    plt.show()

## 5 · FHR + TD3 internals - one figure per diagnostic

`train_diagnostics.csv` is one row per gradient burst; `F.prime_diag_cache()`
bins each run once (per-bin median, cached next to the run), so every panel is
a few hundred points per curve. Each diagnostic is its own figure: the
weighted penalty, the critic TD loss, the recurrence residual, the companion
spectral radius and $\sum c$ (recurrence health), $c(s,a)$ spread on the
state-conditioned arms, the actor loss, the penalty batch size and NaN skips.
The stream ratios have their own section (5b).

In [12]:
F.prime_diag_cache()      # first call: a few seconds per run. Then cached.
STREAM = {"rho", "grad_rho", "loss_ratio", "grad_ratio", "grad_cos",
          "grad_norm_td", "grad_norm_pen"}
for col, fig in F.fig_internals_individual():
    if col in STREAM:
        plt.close(fig)
        continue
    F.save(fig, f"05_{col}")
    plt.show()

diagnostics cache ready (0.5s)


saved: figures/sb3_pendulum_td3/05_penalty_weighted.pdf  figures/sb3_pendulum_td3/05_penalty_weighted.png


saved: figures/sb3_pendulum_td3/05_td_loss.pdf  figures/sb3_pendulum_td3/05_td_loss.png


saved: figures/sb3_pendulum_td3/05_residual_rms.pdf  figures/sb3_pendulum_td3/05_residual_rms.png


saved: figures/sb3_pendulum_td3/05_companion_radius.pdf  figures/sb3_pendulum_td3/05_companion_radius.png


...

saved: figures/sb3_pendulum_td3/05_sum_c.pdf  figures/sb3_pendulum_td3/05_sum_c.png


saved: figures/sb3_pendulum_td3/05_c_spread.pdf  figures/sb3_pendulum_td3/05_c_spread.png


saved: figures/sb3_pendulum_td3/05_actor_loss.pdf  figures/sb3_pendulum_td3/05_actor_loss.png


saved: figures/sb3_pendulum_td3/05_b_h.pdf  figures/sb3_pendulum_td3/05_b_h.png


saved: figures/sb3_pendulum_td3/05_nan_skips.pdf  figures/sb3_pendulum_td3/05_nan_skips.png


## 5b · The two streams - rho by losses and by gradients

How hard each loss term pushes the critic, measured on the same batch:

* $\rho_{loss} = \lambda\cdot$penalty / TD and its unweighted form
  penalty / TD - the ratio of the two loss terms;
* $\rho_{grad} = \lambda\,\|\nabla_\theta$penalty$\| / \|\nabla_\theta$TD$\|$
  and its unweighted form - the ratio of the two **gradient** streams on the
  critic parameters (`agent.grad_probe_every`), plus their cosine (negative =
  the streams conflict) and the two norms.

The unweighted ratios are drawn for the **baseline too**: there the penalty
never enters the loss, so its ratio is the free calibration signal, and the
table converts it into the $\lambda$ that would put a target ratio on the
critic - by gradients (the scale-free choice) and by losses. This is the
quantity that transfers between algorithms whose TD losses sit on different
scales; $\lambda$ itself does not.

In [13]:
for col, fig in F.fig_internals_individual():
    if col not in STREAM:
        plt.close(fig)
        continue
    F.save(fig, f"05b_rho_{col}")
    plt.show()

saved: figures/sb3_pendulum_td3/05b_rho_rho.pdf  figures/sb3_pendulum_td3/05b_rho_rho.png


saved: figures/sb3_pendulum_td3/05b_rho_grad_rho.pdf  figures/sb3_pendulum_td3/05b_rho_grad_rho.png


saved: figures/sb3_pendulum_td3/05b_rho_loss_ratio.pdf  figures/sb3_pendulum_td3/05b_rho_loss_ratio.png


saved: figures/sb3_pendulum_td3/05b_rho_grad_ratio.pdf  figures/sb3_pendulum_td3/05b_rho_grad_ratio.png


saved: figures/sb3_pendulum_td3/05b_rho_grad_cos.pdf  figures/sb3_pendulum_td3/05b_rho_grad_cos.png


saved: figures/sb3_pendulum_td3/05b_rho_grad_norm_td.pdf  figures/sb3_pendulum_td3/05b_rho_grad_norm_td.png


saved: figures/sb3_pendulum_td3/05b_rho_grad_norm_pen.pdf  figures/sb3_pendulum_td3/05b_rho_grad_norm_pen.png


In [14]:
F.table_rho()
print()
RHO = F.table_rho_streams(tail=0.5, targets=(0.1, 0.5, 1.0))

arm                               median TD   median L*pen        rho
----------------------------------------------------------------------
baseline (stock SB3 TD3)              3.542              0          0
FHR lam=0.1 r=2                       4.097         0.1467     0.0358
FHR lam=1 r=2                          3.07           1.15      0.374
FHR lam=1 r=2 frozen-c                3.463          1.155      0.334
FHR lam=0.1 r=2 c(s,a) shared          4.64          4.424      0.953
FHR lam=0.1 r=2 +PER                 0.7251          0.222      0.306

stream ratios over the last 50% of training - Pendulum-v1, seeds 44, 66
arm                                     pen/TD        rho_loss  |g_pen|/|g_TD|        rho_grad             cos
----------------------------------------------------------------------------------------------------------------
baseline (stock SB3 TD3)                0.4164               0          0.2179               0         0.06683
FHR lam=0.1 r=2                

## 6 · Rollout Hankel rank - the critic trace and the policy itself

Stacked per-rollout Hankels of the min-twin critic trace $Q(s_t, \pi(s_t))$
and of each of the 1 action dimension(s) of $\pi(s_t)$, from the
**converged** deterministic policy (one seed per arm). A rank-$r$ Hankel
sequence satisfies an order-$r$ recurrence, so the measured rank of a
converged policy is the smallest order the penalty can enforce without
fighting the solution. One figure per signal.

In [15]:
_ = F.table_rollout_hankel(runner)
fig = F.fig_rollout_hankel_single(runner, which="q")
F.save(fig, "06_rollout_hankel_q")
plt.show()
for j in range(1):
    fig = F.fig_rollout_hankel_single(runner, which="pi", dim=j)
    F.save(fig, f"06_rollout_hankel_pi{j}")
    plt.show()

arm                            rank(Q)  ranks(pi dims)   (energy rank @ 99.9%, seed 44, 3 rollouts)
------------------------------------------------------------------------
baseline (stock SB3 TD3)             7  [41]
FHR lam=0.1 r=2                      8  [43]
FHR lam=1 r=2                        9  [57]
FHR lam=1 r=2 frozen-c               9  [53]
FHR lam=0.1 r=2 c(s,a) shared        7  [72]
FHR lam=0.1 r=2 +PER                 7  [75]


saved: figures/sb3_pendulum_td3/06_rollout_hankel_q.pdf  figures/sb3_pendulum_td3/06_rollout_hankel_q.png


saved: figures/sb3_pendulum_td3/06_rollout_hankel_pi0.pdf  figures/sb3_pendulum_td3/06_rollout_hankel_pi0.png


## 7 · Penalised-window Hankel rank - the in-training probe

Rank measured **where the penalty is applied**: on sampled replay windows
(anchor + `window_rank_lags` same-episode predecessors, online critics,
buffer actions). The probe runs for every arm **including the baseline**,
which measures the same windows, so its curve is the control: if FHR operates
as a rank constraint its arms push the window rank and the penalty-block tail
ratio *below* the baseline on exactly these windows. One figure per arm per
metric, then the all-arm overlay per metric. (In-training measurement - it
cannot be back-filled; `F.window_probe_status()` says which runs carry it.)

In [16]:
_ = F.window_probe_status()

config_sb3_td3.yaml: window_rank_every = 200  (ON)
  baseline (stock SB3 TD3)       2/2 runs carry window_hankel.csv
  FHR lam=0.1 r=2                2/2 runs carry window_hankel.csv
  FHR lam=1 r=2                  2/2 runs carry window_hankel.csv
  FHR lam=1 r=2 frozen-c         2/2 runs carry window_hankel.csv
  FHR lam=0.1 r=2 c(s,a) shared  2/2 runs carry window_hankel.csv
  FHR lam=0.1 r=2 +PER           2/2 runs carry window_hankel.csv


In [17]:
if F.has_window_probe:
    for mkey, mtitle, mscale in F.WINDOW_KEYS:
        for a in F.fhr_arms:
            fig = F.fig_window_rank(a.key, keys=[(mkey, mtitle, mscale)])
            F.save(fig, f"07_window_{mkey}_{a.key}")
            plt.show()
        fig = F.fig_window_rank_overlay(keys=[(mkey, mtitle, mscale)])
        F.save(fig, f"07_window_{mkey}_all")
        plt.show()
    print()
    F.table_window_rank()

saved: figures/sb3_pendulum_td3/07_window_rank999_exp1.pdf  figures/sb3_pendulum_td3/07_window_rank999_exp1.png


saved: figures/sb3_pendulum_td3/07_window_rank999_exp2.pdf  figures/sb3_pendulum_td3/07_window_rank999_exp2.png


saved: figures/sb3_pendulum_td3/07_window_rank999_exp5.pdf  figures/sb3_pendulum_td3/07_window_rank999_exp5.png


saved: figures/sb3_pendulum_td3/07_window_rank999_exp3.pdf  figures/sb3_pendulum_td3/07_window_rank999_exp3.png


saved: figures/sb3_pendulum_td3/07_window_rank999_exp4.pdf  figures/sb3_pendulum_td3/07_window_rank999_exp4.png


...

saved: figures/sb3_pendulum_td3/07_window_pen_tail_ratio_exp3.pdf  figures/sb3_pendulum_td3/07_window_pen_tail_ratio_exp3.png


saved: figures/sb3_pendulum_td3/07_window_pen_tail_ratio_exp4.pdf  figures/sb3_pendulum_td3/07_window_pen_tail_ratio_exp4.png


saved: figures/sb3_pendulum_td3/07_window_pen_tail_ratio_all.pdf  figures/sb3_pendulum_td3/07_window_pen_tail_ratio_all.png

arm                             rank@99.9%   rank@99%    s2/s1   pen tail   (final-quarter means)
----------------------------------------------------------------------------
baseline (stock SB3 TD3)              3.51       2.00   0.1118     0.0052
FHR lam=0.1 r=2                       3.78       1.98   0.1118     0.0056


FHR lam=1 r=2                         3.55       2.00   0.1096     0.0051
FHR lam=1 r=2 frozen-c                3.46       1.99   0.1098     0.0048
FHR lam=0.1 r=2 c(s,a) shared         3.61       2.00   0.1144     0.0061


FHR lam=0.1 r=2 +PER                  3.78       2.00   0.1279     0.0070


## 8 · Compute cost

In [18]:
import os
rows = []
for k, a in F.arms.items():
    for s, d in F.run_dirs(k):
        ck = d / "checkpoints" / "final.pt"
        if ck.exists():
            rows.append((a.plain, s, (os.path.getmtime(ck)
                                      - os.path.getmtime(d / "config.yaml")) / 60))
for label, s, mins in rows:
    print(f"{label:30s} seed {s}: {mins:6.1f} min")
base_name = F.baseline.plain if F.baseline else None
base = [m for l, _, m in rows if l == base_name]
fhr = [m for l, _, m in rows if l != base_name]
if base and fhr:
    print(f"\nbaseline mean {np.mean(base):.1f} min; FHR-arm mean "
          f"{np.mean(fhr):.1f} min (overhead x{np.mean(fhr)/np.mean(base):.2f})")

baseline (stock SB3 TD3)       seed 44:   16.9 min
baseline (stock SB3 TD3)       seed 66:   17.2 min
FHR lam=0.1 r=2                seed 44:   19.9 min
FHR lam=0.1 r=2                seed 66:   19.5 min
FHR lam=1 r=2                  seed 44:   20.0 min
FHR lam=1 r=2                  seed 66:   20.0 min
FHR lam=1 r=2 frozen-c         seed 44:   19.5 min
FHR lam=1 r=2 frozen-c         seed 66:   19.5 min
FHR lam=0.1 r=2 c(s,a) shared  seed 44:   20.2 min
FHR lam=0.1 r=2 c(s,a) shared  seed 66:   20.2 min
FHR lam=0.1 r=2 +PER           seed 44:   20.7 min
FHR lam=0.1 r=2 +PER           seed 66:   20.8 min

baseline mean 17.0 min; FHR-arm mean 20.0 min (overhead x1.17)


## 9 · Final policy rollouts - videos

`record_final_videos` reloads each run's **final checkpoint**, rebuilds the
same env wrapper stack with `render_mode="rgb_array"`, and rolls **one greedy
episode** of the deterministic actor (the policy the `eval.csv` curves score)
through gymnasium's `RecordVideo` -> `<run_dir>/videos/epfinal-episode-0.mp4`.

In [19]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # headless MuJoCo rendering
from IPython.display import Video, display
from run_sb3_seeds import record_final_videos

for k, a in F.arms.items():
    try:
        vids = record_final_videos(k, config=CONFIG)
    except RuntimeError as e:   # arm not finished in this config's manifest yet
        print(f"{a.plain} ({k}): skipped - {e}")
        continue
    for seed, path in vids:
        print(f"{a.plain} ({k}) - seed {seed}: {path}")
        display(Video(str(path), embed=True, width=420))

/home/azureuser/localfiles/Low-Rank-RL-TL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_baseline_seed44_20260903-213144/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory


ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


/home/azureuser/localfiles/Low-Rank-RL-TL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_baseline_seed66_20260903-213144/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such fi

baseline (stock SB3 TD3) (baseline) - seed 44: /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_baseline_seed44_20260903-213144/videos/epfinal-episode-0.mp4


baseline (stock SB3 TD3) (baseline) - seed 66: /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_baseline_seed66_20260903-213144/videos/epfinal-episode-0.mp4


...

FHR lam=0.1 r=2 c(s,a) shared (exp3) - seed 66: /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_exp3_seed66_20260903-213153/videos/epfinal-episode-0.mp4


/home/azureuser/localfiles/Low-Rank-RL-TL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_exp4_seed44_20260903-213144/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file o

/home/azureuser/localfiles/Low-Rank-RL-TL/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_exp4_seed66_20260903-213144/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file o

FHR lam=0.1 r=2 +PER (exp4) - seed 44: /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_exp4_seed44_20260903-213144/videos/epfinal-episode-0.mp4


FHR lam=0.1 r=2 +PER (exp4) - seed 66: /mnt/batch/tasks/shared/LS_root/mounts/clusters/souparna20031/code/Users/souparna2003/Low-Rank-RL-TL/experiments/stable_baselines_3/pendulum/cached/runs/sb3_pendulum_td3_exp4_seed66_20260903-213144/videos/epfinal-episode-0.mp4


## 10 · Findings

*(fill in after the family completes)*